# 1x4 AWG demultiplexer

Schematic reconstruction after Smit and van Dam, IEEE J. Sel. Top.
Quantum Electron. 1996
([doi:10.1109/2944.577370](https://doi.org/10.1109/2944.577370)).

Chain the input coupler onto `i1`. Fan the four detectors out
at 1.5 times the rise, with pitch from the detector height.
`o1` is the lowest output.


In [1]:
from optodraw import Scene

def half_size(component):
    probe = Scene(title="", width=2000, height=2000, platform="on_chip")
    probe.add(component)
    x0, y0, x1, y1 = probe.content_bounds()
    return 0.5 * (x1 - x0), 0.5 * (y1 - y0)


In [2]:
def dangling_optical(scene, connected):
    used = {(id(comp), name) for comp, name in connected}
    leftover = []
    for component in scene.walk():
        ports = getattr(component, "ports", None)
        if not callable(ports):
            continue
        for name, port in ports().items():
            if getattr(port, "kind", "optical") in ("electrical", "fiber"):   # a fibre port is the off-chip side
                continue
            if name.startswith(("e", "h", "gnd", "sig")):
                continue
            if (id(component), name) in used:
                continue
            label = getattr(component, "label", None) or component.__class__.__name__
            leftover.append(f"{label}:{name}")
    return leftover


In [3]:
from optodraw import Scene, AWG, VerticalGratingCoupler, OnChipPhotodetector
from optodraw.layout import chain_ports

pd_hh = half_size(OnChipPhotodetector(label="PD"))[1]
PITCH = max(2.2 * pd_hh, 32.0)
HOP = 20.0
print("PD half height", pd_hh, "pitch", PITCH)


PD half height 15.0 pitch 33.0


In [4]:
scene = Scene(title="1x4 AWG demux (after Smit 1996)", width=520, height=240, platform="on_chip")
gc = scene.add(VerticalGratingCoupler(center=(0, 0), angle=180, label="in"))
awg = scene.add(AWG(channel_count=4, array_count=8, label="AWG"))
chain_ports([(gc, "o1", awg, "i1")], gap=HOP, grid=0)
n = 4
outs = [awg.world_ports()[f"o{i}"] for i in range(1, n + 1)]
pd_ys = [(i - 1.5) * PITCH for i in range(n)]
rise = max(abs(pd_ys[i] - float(outs[i].center.y)) for i in range(n))
gap_out = max(1.5 * rise, 48.0)
x_port = max(float(port.center.x) for port in outs) + gap_out
local_x = float(OnChipPhotodetector(label="PD").ports()["o1"].center.x)
pds = [
    scene.add(OnChipPhotodetector(center=(x_port - local_x, pd_ys[i]), label=f"PD{i + 1}",
                                   label_side="right"))
    for i in range(n)
]
scene.route((gc, "o1"), (awg, "i1"), kind="waveguide", path_type="smooth")
connected = [(gc, "o1"), (awg, "i1")]
for i, pd in enumerate(pds, start=1):
    scene.route((awg, f"o{i}"), (pd, "o1"), kind="waveguide", path_type="smooth")
    connected.extend([(awg, f"o{i}"), (pd, "o1")])
print("unconnected", dangling_optical(scene, connected))
scene.fit_frame(margin=16)


unconnected []


In [5]:
print("bounds", scene.content_bounds())
scene.preview(show_3d=False)


bounds (np.float64(-20.7975), np.float64(-64.5), np.float64(393.5410169955413), np.float64(132.3419748385695))


ScenePreview('1x4 AWG demux (after Smit 1996)', 900x520, 3d=off)